In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. VERİYİ YÜKLEYELİM
# ============================================================
train = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/test_x.csv')

print("Train shape:", train.shape)
print("Test shape :", test.shape)

TARGET    = 'bilissel_performans_skoru'
DROP_COLS = ['id', TARGET]

test_ids     = test['id'].copy()
train_target = train[TARGET].copy()

train['is_train'] = 1
test['is_train']  = 0
test[TARGET]      = np.nan

all_data = pd.concat([train, test], ignore_index=True)

# ============================================================
# 2. PİPELINE
# ============================================================
country_map = {
    'Spain'      : 'Ispanya',
    'South Korea': 'Guney Kore',
    'Sweden'     : 'Isvec',
    'Mexico'     : 'Meksika',
    'Netherlands': 'Hollanda'
}
all_data['ulke'] = all_data['ulke'].replace(country_map)
all_data = all_data.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
all_data = all_data.replace(['nan', 'NaN', '?', 'None', '', 'null', 'NULL'], np.nan)

all_data['meslek']             = all_data['meslek'].fillna('Bilinmiyor')
all_data['kronotip']           = all_data['kronotip'].fillna(all_data['kronotip'].mode()[0])
all_data['ruh_sagligi_durumu'] = all_data['ruh_sagligi_durumu'].fillna('Bilinmiyor')
all_data['vucut_kitle_indeksi']   = all_data['vucut_kitle_indeksi'].fillna(all_data['vucut_kitle_indeksi'].median())
all_data['uyku_oncesi_kafein_mg'] = all_data['uyku_oncesi_kafein_mg'].fillna(0)

all_data['stres_skoru'] = all_data.groupby('ruh_sagligi_durumu')['stres_skoru'].transform(
    lambda x: x.fillna(x.median())
)
all_data['stres_skoru'] = all_data['stres_skoru'].fillna(all_data['stres_skoru'].median())

num_cols = all_data.select_dtypes(include=[np.number]).columns
all_data[num_cols] = all_data[num_cols].fillna(all_data[num_cols].median())

outlier_cols = ['vucut_kitle_indeksi', 'uyku_oncesi_kafein_mg',
                'gunluk_adim_sayisi', 'sekerleme_suresi_dk',
                'gunluk_calisma_saati', 'uykuya_dalma_suresi_dk']
for col in outlier_cols:
    lower = all_data[col].quantile(0.01)
    upper = all_data[col].quantile(0.99)
    all_data[col] = all_data[col].clip(lower=lower, upper=upper)

print(f"Pipeline tamam! Eksik değer: {all_data.isnull().sum().sum()}")

# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================
cat_cols = ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']

# UYKU KALİTESİ
all_data['toplam_onarici_uyku']        = all_data['rem_yuzdesi'] + all_data['derin_uyku_yuzdesi']
all_data['onarici_uyku_yeterli']       = (all_data['toplam_onarici_uyku'] > 45).astype(int)
all_data['rem_derin_denge']            = all_data['rem_yuzdesi'] / (all_data['derin_uyku_yuzdesi'] + 1)
all_data['uyku_surekliligi_bozuklugu'] = all_data['uykuya_dalma_suresi_dk'] * all_data['gecelik_uyanma_sayisi']
all_data['derin_uyku_surekliligi']     = all_data['derin_uyku_yuzdesi'] / (all_data['gecelik_uyanma_sayisi'] + 1)
all_data['rem_surekliligi']            = all_data['rem_yuzdesi'] / (all_data['gecelik_uyanma_sayisi'] + 1)

# İYİLEŞME İNDEKSİ
all_data['iyilesme_indeksi'] = all_data['toplam_onarici_uyku'] / (all_data['stres_skoru'] + 1)
all_data['guclu_iyilesme']   = (
    all_data['toplam_onarici_uyku'] /
    (all_data['stres_skoru'] + 1) /
    (all_data['gecelik_uyanma_sayisi'] + 1)
)
all_data['uyku_verimliligi'] = all_data['toplam_onarici_uyku'] / (all_data['uykuya_dalma_suresi_dk'] + 1)
all_data['stres_direnci']    = (
    all_data['toplam_onarici_uyku'] * all_data['gunluk_adim_sayisi'] /
    (all_data['stres_skoru'] + 1)
)
all_data['yas_iyilesme'] = all_data['yas'] * all_data['iyilesme_indeksi']

# KRONOTİP
def kronotip_senkron(row):
    kronotip = str(row['kronotip']).lower()
    gun      = str(row['gun_tipi']).lower()
    if 'sabah' in kronotip and 'ici' in gun:  return 2
    if 'gece'  in kronotip and 'sonu' in gun: return 2
    if 'notr'  in kronotip:                   return 1
    return 0

all_data['kronotip_senkron']  = all_data.apply(kronotip_senkron, axis=1)
all_data['kronotip_gun_tipi'] = all_data['kronotip'].astype(str) + '_' + all_data['gun_tipi'].astype(str)

# KAFEİN
def kafein_etkisi(mg):
    if mg == 0:     return 0
    elif mg <= 40:  return 1
    elif mg <= 200: return 2
    else:           return 3

all_data['kafein_kategori']     = all_data['uyku_oncesi_kafein_mg'].apply(kafein_etkisi)
all_data['kafein_uyku_latency'] = all_data['uyku_oncesi_kafein_mg'] * all_data['uykuya_dalma_suresi_dk']
all_data['ekran_kafein']        = all_data['uyku_oncesi_ekran_suresi_dk'] * all_data['uyku_oncesi_kafein_mg']

# RUH SAĞLIĞI
def ruh_sagligi_riski(durum):
    durum = str(durum).lower()
    if 'saglikli' in durum: return 0
    elif 'anksiyete' in durum and 'depresyon' in durum: return 3
    elif 'anksiyete' in durum: return 2
    elif 'depresyon' in durum: return 2
    else: return 1

all_data['ruh_sagligi_riski']     = all_data['ruh_sagligi_durumu'].apply(ruh_sagligi_riski)
all_data['ruh_uyku_bilesik_risk'] = all_data['ruh_sagligi_riski'] * all_data['uyku_surekliligi_bozuklugu']
all_data['ruh_stres_riski']       = all_data['ruh_sagligi_riski'] * all_data['stres_skoru']
all_data['ruh_stres_combo']       = (
    all_data['ruh_sagligi_durumu'].astype(str) + '_' +
    pd.cut(all_data['stres_skoru'], bins=3, labels=['dusuk', 'orta', 'yuksek']).astype(str)
)

# STRES
all_data['zihinsel_yuk']         = all_data['gunluk_calisma_saati'] * all_data['stres_skoru']
all_data['stres_aktivite_orani'] = all_data['stres_skoru'] / (all_data['gunluk_adim_sayisi'] / 1000 + 1)
all_data['yas_stres']            = all_data['yas'] * all_data['stres_skoru']
all_data['stres_nabiz']          = all_data['stres_skoru'] * all_data['dinlenik_nabiz_bpm']

# ODA SICAKLIĞI
all_data['ideal_sicaklik_farki'] = (all_data['oda_sicakligi_celsius'] - 19).abs()
all_data['sicak_oda']            = (all_data['oda_sicakligi_celsius'] > 21).astype(int)
all_data['soguk_oda']            = (all_data['oda_sicakligi_celsius'] < 17).astype(int)
all_data['sicaklik_kafein']      = all_data['ideal_sicaklik_farki'] * all_data['uyku_oncesi_kafein_mg']
all_data['sicaklik_uyanma']      = all_data['ideal_sicaklik_farki'] * all_data['gecelik_uyanma_sayisi']

# FİZİKSEL
all_data['adim_calisma_orani']   = all_data['gunluk_adim_sayisi'] / (all_data['gunluk_calisma_saati'] + 1)
all_data['sekerlemedeki_telafi'] = all_data['sekerleme_suresi_dk'] * all_data['gecelik_uyanma_sayisi']
all_data['sosyal_jetlag']        = all_data['hafta_sonu_uyku_farki_saat'].abs()
all_data['toplam_uyku_bozucu']   = (
    all_data['uyku_oncesi_kafein_mg'] +
    all_data['uyku_oncesi_ekran_suresi_dk'] +
    all_data['gecelik_uyanma_sayisi'] * 10 +
    all_data['sosyal_jetlag'] * 5 +
    all_data['ideal_sicaklik_farki'] * 2
)

def yas_grubu(yas):
    if yas < 30:   return 0
    elif yas < 50: return 1
    else:          return 2

def vki_kategori(vki):
    if vki < 18.5: return 0
    elif vki < 25: return 1
    elif vki < 30: return 2
    else:          return 3

all_data['yas_grubu']    = all_data['yas'].apply(yas_grubu)
all_data['vki_kategori'] = all_data['vucut_kitle_indeksi'].apply(vki_kategori)
all_data['yuksek_nabiz'] = (all_data['dinlenik_nabiz_bpm'] > 80).astype(int)
all_data['vki_aktivite'] = all_data['vucut_kitle_indeksi'] / (all_data['gunluk_adim_sayisi'] / 1000 + 1)

# POLİNOMYAL ETKİLEŞİMLER
all_data['iyilesme_x_ruh']       = all_data['iyilesme_indeksi'] * all_data['ruh_sagligi_riski']
all_data['iyilesme_x_stres']     = all_data['iyilesme_indeksi'] * all_data['stres_skoru']
all_data['guclu_iyilesme_x_ruh'] = all_data['guclu_iyilesme']  * all_data['ruh_sagligi_riski']
all_data['psqi_benzeri_skor']    = (
    all_data['uykuya_dalma_suresi_dk'] / 15 +
    all_data['gecelik_uyanma_sayisi'] +
    all_data['uyku_oncesi_kafein_mg'] / 100 +
    all_data['sosyal_jetlag']
)
all_data['biyolojik_yas_proxy']  = (
    all_data['dinlenik_nabiz_bpm'] * 0.5 +
    all_data['vucut_kitle_indeksi'] * 0.3 +
    all_data['stres_skoru'] * 2
)
all_data['net_iyilesme']         = (
    all_data['toplam_onarici_uyku'] +
    all_data['gunluk_adim_sayisi'] / 1000 -
    all_data['stres_skoru'] -
    all_data['psqi_benzeri_skor']
)
all_data['uyku_borcu']           = all_data['hafta_sonu_uyku_farki_saat'] * all_data['gecelik_uyanma_sayisi']
all_data['senkron_iyilesme']     = all_data['kronotip_senkron'] * all_data['iyilesme_indeksi']
all_data['nabiz_iyilesme']       = all_data['iyilesme_indeksi'] / (all_data['dinlenik_nabiz_bpm'] / 60)
all_data['rem_kare']             = all_data['rem_yuzdesi'] ** 2
all_data['rem_x_derin']          = all_data['rem_yuzdesi'] * all_data['derin_uyku_yuzdesi']
all_data['rem_x_stres']          = all_data['rem_yuzdesi'] * all_data['stres_skoru']
all_data['rem_x_uyanma']         = all_data['rem_yuzdesi'] * all_data['gecelik_uyanma_sayisi']
all_data['sicaklik_kare']        = all_data['ideal_sicaklik_farki'] ** 2
all_data['sicaklik_x_iyilesme']  = all_data['ideal_sicaklik_farki'] * all_data['iyilesme_indeksi']
all_data['stres_kare']           = all_data['stres_skoru'] ** 2
all_data['stres_x_iyilesme']     = all_data['stres_skoru'] * all_data['iyilesme_indeksi']
all_data['uyku_telafi_orani']    = all_data['hafta_sonu_uyku_farki_saat'] / (all_data['gecelik_uyanma_sayisi'] + 1)
all_data['aktivite_denge']       = all_data['gunluk_adim_sayisi'] / 1000 - all_data['gunluk_calisma_saati']
all_data['mevsim_kronotip']      = all_data['mevsim'].astype(str) + '_' + all_data['kronotip'].astype(str)

cat_cols_extended = cat_cols + ['kronotip_gun_tipi', 'ruh_stres_combo', 'mevsim_kronotip']

print(f"Feature sayısı: {all_data.shape[1]}")

# ============================================================
# 4. GRUP İSTATİSTİKLERİ
# ============================================================
train_stats = all_data[all_data['is_train'] == 1].copy()

ulke_stats = train_stats.groupby('ulke').agg(
    ulke_ort_stres = ('stres_skoru',        'mean'),
    ulke_ort_uyku  = ('toplam_onarici_uyku', 'mean'),
    ulke_ort_adim  = ('gunluk_adim_sayisi',  'mean'),
).reset_index()

meslek_stats = train_stats.groupby('meslek').agg(
    meslek_ort_stres   = ('stres_skoru',         'mean'),
    meslek_ort_calisma = ('gunluk_calisma_saati', 'mean'),
    meslek_ort_uyku    = ('toplam_onarici_uyku',  'mean'),
).reset_index()

ruh_stats = train_stats.groupby('ruh_sagligi_durumu').agg(
    ruh_ort_stres = ('stres_skoru',        'mean'),
    ruh_ort_uyku  = ('toplam_onarici_uyku', 'mean'),
).reset_index()

all_data = all_data.merge(ulke_stats,   on='ulke',               how='left')
all_data = all_data.merge(meslek_stats, on='meslek',             how='left')
all_data = all_data.merge(ruh_stats,    on='ruh_sagligi_durumu', how='left')

for col in ['ulke_ort_stres', 'ulke_ort_uyku', 'ulke_ort_adim',
            'meslek_ort_stres', 'meslek_ort_calisma', 'meslek_ort_uyku',
            'ruh_ort_stres', 'ruh_ort_uyku']:
    all_data[col].fillna(all_data[col].median(), inplace=True)

# ============================================================
# 5. TARGET ENCODING
# ============================================================
train_part = all_data[all_data['is_train'] == 1].copy()
test_part  = all_data[all_data['is_train'] == 0].copy()

target_encode_cols = [
    'ruh_sagligi_durumu', 'gun_tipi', 'meslek', 'kronotip',
    'mevsim', 'ulke', 'cinsiyet',
    'kronotip_gun_tipi', 'ruh_stres_combo', 'mevsim_kronotip'
]
kf_te = KFold(n_splits=5, shuffle=True, random_state=42)

for col in target_encode_cols:
    new_col = f'{col}_target_enc'
    train_part[new_col] = np.nan
    for tr_idx, val_idx in kf_te.split(train_part):
        fold_mean = train_part.iloc[tr_idx].groupby(col)[TARGET].mean()
        train_part.iloc[val_idx, train_part.columns.get_loc(new_col)] = (
            train_part.iloc[val_idx][col].map(fold_mean)
        )
    global_mean = train_part.groupby(col)[TARGET].mean()
    test_part[new_col] = test_part[col].map(global_mean)
    global_avg = train_part[TARGET].mean()
    train_part[new_col].fillna(global_avg, inplace=True)
    test_part[new_col].fillna(global_avg, inplace=True)

all_data_encoded = pd.concat([train_part, test_part], ignore_index=True)
cat_cols_extended = cat_cols + ['kronotip_gun_tipi', 'ruh_stres_combo', 'mevsim_kronotip']

le = LabelEncoder()
for col in cat_cols_extended:
    all_data_encoded[col] = le.fit_transform(all_data_encoded[col].astype(str))
    all_data[col]         = le.fit_transform(all_data[col].astype(str))

# ============================================================
# 6. VERİ SETLERİNİ AYIR
# ============================================================
train_enc  = all_data_encoded[all_data_encoded['is_train'] == 1].drop('is_train', axis=1)
test_enc   = all_data_encoded[all_data_encoded['is_train'] == 0].drop('is_train', axis=1)
X_enc      = train_enc.drop(DROP_COLS, axis=1)
X_test_enc = test_enc.drop(DROP_COLS, axis=1)

train_orig  = all_data[all_data['is_train'] == 1].drop('is_train', axis=1)
test_orig   = all_data[all_data['is_train'] == 0].drop('is_train', axis=1)
X_orig      = train_orig.drop(DROP_COLS, axis=1)
X_test_orig = test_orig.drop(DROP_COLS, axis=1)

y  = train_enc[TARGET]
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print(f"Encoded feature sayısı : {X_enc.shape[1]}")
print(f"Original feature sayısı: {X_orig.shape[1]}")

# ============================================================
# 7. OPTUNA — CatBoost Tuning (100 trial)
# ============================================================
print("\n=== Optuna CatBoost Tuning (100 trial) ===")

cat_features_idx = [X_enc.columns.get_loc(col) for col in cat_cols_extended if col in X_enc.columns]

def objective(trial):
    params = {
        'iterations'          : trial.suggest_int('iterations', 1000, 6000),
        'learning_rate'       : trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'depth'               : trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg'         : trial.suggest_float('l2_leaf_reg', 1, 15),
        'bagging_temperature' : trial.suggest_float('bagging_temperature', 0, 1),
        'random_strength'     : trial.suggest_float('random_strength', 0, 2),
        'border_count'        : trial.suggest_int('border_count', 32, 255),
        'random_seed'         : 42,
        'eval_metric'         : 'RMSE',
        'early_stopping_rounds': 100,
        'verbose'             : 0,
    }
    oof = np.zeros(len(X_enc))
    for tr_idx, val_idx in kf.split(X_enc):
        X_tr, X_val = X_enc.iloc[tr_idx], X_enc.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model = CatBoostRegressor(**params, cat_features=cat_features_idx)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        oof[val_idx] = model.predict(X_val)
    return np.sqrt(mean_squared_error(y, oof))

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30, show_progress_bar=True)

best_params = study.best_params
print(f"\n✅ En iyi Optuna RMSE: {study.best_value:.4f}")
print(f"En iyi parametreler: {best_params}")

# ============================================================
# 8. SEED AVERAGING (10 seed)
# ============================================================
SEEDS = [42, 123, 2024, 7, 999, 1234, 555, 888, 314, 271]

# CatBoost Seed Averaging (Optuna parametreleriyle)
print("\n=== CatBoost Seed Averaging (10 seed, Optuna params) ===")
cb_seed_oof        = np.zeros(len(X_enc))
cb_seed_test_preds = np.zeros(len(X_test_enc))

for seed in SEEDS:
    print(f"Seed: {seed}")
    seed_oof   = np.zeros(len(X_enc))
    seed_preds = np.zeros(len(X_test_enc))
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_enc)):
        X_tr, X_val = X_enc.iloc[tr_idx], X_enc.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model = CatBoostRegressor(
            **best_params,
            random_seed=seed,
            eval_metric='RMSE',
            early_stopping_rounds=100,
            verbose=0,
            cat_features=cat_features_idx
        )
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        seed_oof[val_idx] = model.predict(X_val)
        seed_preds       += model.predict(X_test_enc) / kf.n_splits
    seed_rmse = np.sqrt(mean_squared_error(y, seed_oof))
    print(f"  RMSE: {seed_rmse:.4f}")
    cb_seed_oof        += seed_oof   / len(SEEDS)
    cb_seed_test_preds += seed_preds / len(SEEDS)

cb_seed_rmse = np.sqrt(mean_squared_error(y, cb_seed_oof))
print(f"✅ CatBoost Seed Avg RMSE: {cb_seed_rmse:.4f}")

# XGBoost Seed Averaging (10 seed)
print("\n=== XGBoost Seed Averaging (10 seed) ===")
xgb_seed_oof        = np.zeros(len(X_orig))
xgb_seed_test_preds = np.zeros(len(X_test_orig))

for seed in SEEDS:
    print(f"Seed: {seed}")
    seed_oof   = np.zeros(len(X_orig))
    seed_preds = np.zeros(len(X_test_orig))
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_orig)):
        X_tr, X_val = X_orig.iloc[tr_idx], X_orig.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model = xgb.XGBRegressor(
            objective='reg:squarederror', learning_rate=0.02,
            max_depth=6, min_child_weight=30,
            subsample=0.7, colsample_bytree=0.7,
            reg_alpha=0.1, reg_lambda=0.1,
            n_estimators=5000, tree_method='hist',
            random_state=seed, verbosity=0,
            early_stopping_rounds=100
        )
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)
        seed_oof[val_idx] = model.predict(X_val)
        seed_preds       += model.predict(X_test_orig) / kf.n_splits
    xgb_seed_oof        += seed_oof   / len(SEEDS)
    xgb_seed_test_preds += seed_preds / len(SEEDS)

xgb_seed_rmse = np.sqrt(mean_squared_error(y, xgb_seed_oof))
print(f"✅ XGBoost Seed Avg RMSE: {xgb_seed_rmse:.4f}")

# ============================================================
# 9. STACKING (CatBoost + XGBoost)
# ============================================================
print("\n=== Stacking (CatBoost + XGBoost) ===")
stack_train = np.column_stack([cb_seed_oof, xgb_seed_oof])
stack_test  = np.column_stack([cb_seed_test_preds, xgb_seed_test_preds])
stack_oof   = np.zeros(len(y))
stack_preds = np.zeros(len(stack_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(stack_train)):
    X_tr, X_val = stack_train[tr_idx], stack_train[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    meta = Ridge(alpha=1.0)
    meta.fit(X_tr, y_tr)
    stack_oof[val_idx] = meta.predict(X_val)
    stack_preds       += meta.predict(stack_test) / kf.n_splits
    print(f"Fold {fold+1} RMSE: {np.sqrt(mean_squared_error(y_val, stack_oof[val_idx])):.4f}")

stack_rmse = np.sqrt(mean_squared_error(y, stack_oof))
print(f"✅ Stacking OOF RMSE: {stack_rmse:.4f}")

meta_final = Ridge(alpha=1.0).fit(stack_train, y)
print("\nMeta model ağırlıkları:")
for name, coef in zip(['CatBoost', 'XGBoost'], meta_final.coef_):
    print(f"  {name:15s}: {coef:.4f}")

# ============================================================
# 10. SONUÇLAR
# ============================================================
print("\n" + "="*50)
print("FINAL KARŞILAŞTIRMA")
print("="*50)
results = {
    'CatBoost Seed Avg': cb_seed_rmse,
    'XGBoost Seed Avg' : xgb_seed_rmse,
    'Stacking'         : stack_rmse,
}
for name, score in sorted(results.items(), key=lambda x: x[1]):
    print(f"  {name:20s}: {score:.4f}")

best_name  = min(results, key=results.get)
best_preds = {
    'CatBoost Seed Avg': cb_seed_test_preds,
    'XGBoost Seed Avg' : xgb_seed_test_preds,
    'Stacking'         : stack_preds,
}[best_name]

print(f"\n🏆 En iyi: {best_name} (RMSE: {results[best_name]:.4f})")

submission = pd.DataFrame({
    'id'                       : test_ids.values,
    'bilissel_performans_skoru': best_preds
})
submission.to_csv('submission.csv', index=False)
print("\n🎉 submission.csv hazır!")
print(submission.head())
print("Submission shape:", submission.shape)

Train shape: (56000, 24)
Test shape : (24000, 23)
Pipeline tamam! Eksik değer: 0
Feature sayısı: 82
Encoded feature sayısı : 97
Original feature sayısı: 87

=== Optuna CatBoost Tuning (100 trial) ===


  0%|          | 0/30 [00:00<?, ?it/s]


✅ En iyi Optuna RMSE: 1.2181
En iyi parametreler: {'iterations': 4834, 'learning_rate': 0.026580622243780702, 'depth': 5, 'l2_leaf_reg': 6.977974914626927, 'bagging_temperature': 0.4274982262557853, 'random_strength': 1.5168781326146181, 'border_count': 101}

=== CatBoost Seed Averaging (10 seed, Optuna params) ===
Seed: 42
  RMSE: 1.2181
Seed: 123
  RMSE: 1.2188
Seed: 2024
  RMSE: 1.2181
Seed: 7
  RMSE: 1.2184
Seed: 999
  RMSE: 1.2184
Seed: 1234
  RMSE: 1.2187
Seed: 555
  RMSE: 1.2186
Seed: 888
  RMSE: 1.2188
Seed: 314
  RMSE: 1.2188
Seed: 271
  RMSE: 1.2187
✅ CatBoost Seed Avg RMSE: 1.2177

=== XGBoost Seed Averaging (10 seed) ===
Seed: 42
Seed: 123
Seed: 2024
Seed: 7
Seed: 999
Seed: 1234
Seed: 555
Seed: 888
Seed: 314
Seed: 271
✅ XGBoost Seed Avg RMSE: 1.2219

=== Stacking (CatBoost + XGBoost) ===
Fold 1 RMSE: 1.2217
Fold 2 RMSE: 1.2201
Fold 3 RMSE: 1.2020
Fold 4 RMSE: 1.2109
Fold 5 RMSE: 1.2323
✅ Stacking OOF RMSE: 1.2174

Meta model ağırlıkları:
  CatBoost       : 0.7838
  XGBoost